#### Is it possible to tell whether a specific kind of textile (protective workwear, bed linen etc) is produced in specific countries mostly?

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import re
import pandas as pd
import os
from rapidfuzz import process, fuzz

In [ ]:
# 1) Load & Preprocess Data

df_ted = pd.read_excel("outputs/ted_only_selected_notices.xlsx")
df_trade = pd.read_excel("outputs/merged_tradeatlas_clean.xlsx")

mask_multi = df_ted["winner_name"].str.contains(r",\s*", na=False)
df_multinames = df_ted[mask_multi]

df_ted = df_ted[~mask_multi]


df_ted = df_ted[
    ["winner_name", "winner_country", "buyer_country", "buyer_town",
     "buyer_type", "publication_date", "main_classification"]
].dropna(subset=["winner_name", "publication_date"])

df_trade = df_trade[
    ["importer_name", "exporter_name", "exporter_country",
     "usd_fob", "arrival_date", "hs_code"]
].dropna(subset=["importer_name", "arrival_date", "hs_code"])

df_ted["publication_date"] = pd.to_datetime(df_ted["publication_date"], errors="coerce")
df_trade["arrival_date"] = pd.to_datetime(df_trade["arrival_date"], errors="coerce")

In [ ]:
def preprocess_name(name):
    if pd.isna(name):
        return ""
    clean = (
        str(name).lower()
        .replace("&", "and")
        .replace(",", "")
        .replace(".", "")
        .replace("gmbh & co kg", "gmbh")
        .replace("co kg", "")
        .strip()
    )
    return " ".join(clean.split()[:3])

df_ted["short_winner"] = df_ted["winner_name"].apply(preprocess_name)
df_trade["short_importer"] = df_trade["importer_name"].apply(preprocess_name)


# 2) SINGLE CLEANING FUNCTION (used everywhere)

def clean_company_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()

    suffixes = [
        r"gmbh", r"gmbh & co\.? kg", r"& co\.? kg", r"co\.? kg",
        r"kg", r"ag", r"se", r"ohg", r"ltd\.?", r"llc",
        r"inc\.?", r"e\.k\.?", r"gmbh & co", r"gmbh \+ co\.? kg",
        r"\+ co\.? kg"
    ]

    pattern = "(" + "|".join(suffixes) + ")"
    name = re.sub(pattern, "", name).strip()

    # remove punctuation / normalize spacing
    name = re.sub(r"[^\w\s\-]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name


# Normalize names ONCE
df_ted["short_winner"] = df_ted["winner_name"].apply(clean_company_name)
df_trade["short_importer"] = df_trade["importer_name"].apply(clean_company_name)


# 3) Multi-strategy fuzzy matching (but simple)

def multi_strategy_match(ted_name, trade_names):

    # simple variants built from ONE normalization function
    variants = [
        ted_name,
        clean_company_name(ted_name),
        " ".join(ted_name.split()[:3]),  # first 3 words
        " ".join(ted_name.split()[:2])   # first 2 words
    ]

    best_match = None
    best_score = 0

    for v in variants:
        match = process.extractOne(v, trade_names, scorer=fuzz.token_set_ratio)
        if match and match[1] > best_score:
            best_match = match[0]
            best_score = match[1]

    return (best_match, best_score) if best_score >= 85 else (None, None)


# 4) FINAL MATCHING FUNCTION (clean + minimal)

def match_by_name(df_ted, df_trade, similarity_threshold=85):

    results = []
    trade_names = df_trade["short_importer"].unique()

    for _, ted_row in df_ted.iterrows():

        name_clean = ted_row["short_winner"]
        if not name_clean:
            continue

        importer_match, score = multi_strategy_match(name_clean, trade_names)

        if importer_match and score >= similarity_threshold:

            matched_rows = df_trade[df_trade["short_importer"] == importer_match]

            for _, trade_row in matched_rows.iterrows():
                results.append({
                    "winner_name": ted_row["winner_name"],
                    "short_winner": ted_row["short_winner"],
                    "winner_country": ted_row["winner_country"],
                    "buyer_country": ted_row["buyer_country"],
                    "buyer_town": ted_row["buyer_town"],
                    "buyer_type": ted_row["buyer_type"],
                    "publication_date": ted_row["publication_date"],
                    "main_classification": ted_row["main_classification"],
                    "importer_name": trade_row["importer_name"],
                    "short_importer": trade_row["short_importer"],
                    "exporter_name": trade_row["exporter_name"],
                    "exporter_country": trade_row["exporter_country"],
                    "hs_code": trade_row["hs_code"],
                    "usd_fob": trade_row["usd_fob"],
                    "arrival_date": trade_row["arrival_date"],
                    "similarity": score
                })

    df_matches = pd.DataFrame(results)

    if not df_matches.empty:
        df_matches["days_difference"] = (
            df_matches["arrival_date"] - df_matches["publication_date"]
        ).abs().dt.days

        df_matches["months_difference"] = (df_matches["days_difference"] / 30).round(1)

    return df_matches


# 5) RUN MATCHING

df_all_matches = match_by_name(df_ted, df_trade)

print("Matched winners:", df_all_matches["winner_name"].nunique())
df_all_matches.head()


In [ ]:
df_all_matches[["winner_name", "importer_name", "similarity"]].drop_duplicates().to_csv("outputs/matched_winners.csv")

In [ ]:
df_all_matches.to_csv("outputs/df_all_matches.csv")

In [ ]:
df_all_matches.winner_name.nunique()

In [ ]:
df_all_matches

In [ ]:
pubdate_counts = (
    df_all_matches
    .groupby("winner_name")["publication_date"]
    .nunique()
    .reset_index(name="unique_publication_dates")
)


In [ ]:
pubdate_counts.winner_name.unique()

In [ ]:
# Filter by time window
def filter_by_time(df_matches, max_days):
    return df_matches[df_matches["days_difference"] <= max_days]

df_6m = filter_by_time(df_all_matches, 180).drop_duplicates()

# --- Textile HS code mapping ---
hs_mapping = {
    "610510": "Men’s cotton shirts",
    "610910": "T-shirts, cotton",
    "611420": "Cotton sweatshirts / pullovers",
    "611610": "Gloves, knitted or crocheted",
    "611692": "Gloves, synthetic fibres",
    "620312": "Men’s suits, synthetic fibres",
    "620319": "Men’s jackets & blazers (other textiles)",
    "620333": "Men’s coats, synthetic fibres",
    "620729": "Blouses, man-made fibres",
    "621010": "Protective garments (industrial / medical)",
    "621133": "Men’s coats, man-made fibres",
    "630231": "Bed linen, cotton",
    "630260": "Toilet and kitchen linen, terry fabric",
    "520852": "Woven cotton fabrics (dyed)",
    "540752": "Woven synthetic fabrics (dyed)",
    "640610": "Protective footwear (textile upper)",
    "420291": "Textile gloves and similar accessories",
    "420329": "Protective clothing accessories, textile",
}

non_textile = ["847141", "854231"]  # irrelevant base codes

# --- Clean code strings ---
df_6m["hs_code"] = df_6m["hs_code"].astype(str).str.strip()

# --- Filter out non-textile HS codes (if they start with irrelevant prefixes) ---
df_6m = df_6m[~df_6m["hs_code"].str.startswith(tuple(non_textile))].copy()

# --- Map textile HS codes by prefix matching ---
def map_hs_description(code):
    if pd.isna(code):
        return None
    code_str = str(code)
    for base, desc in hs_mapping.items():
        if code_str.startswith(base):
            return desc
    return "Other textile articles"

df_6m["description"] = df_6m["hs_code"].apply(map_hs_description)

In [ ]:
# df_6m.winner_name.value_counts().to_csv("outputs\matched_winners.csv")

In [ ]:
df_6m.winner_name.nunique()

In [ ]:
import urllib.parse
import pycountry

# Convert "Pakistan" → "PK"
def iso2(country):
    try:
        return pycountry.countries.lookup(country).alpha_2
    except:
        return None   # or "XX" if you prefer

# Extract first two words of exporter name
def first_two(name):
    if not isinstance(name, str):
        return ""
    parts = name.split()
    return " ".join(parts[:2])

# Build OSH link
def make_osh_link(exporter_name, exporter_country):
    code = iso2(exporter_country)
    if code is None:
        return None  # skip unknown countries
    
    q = urllib.parse.quote_plus(first_two(exporter_name))
    return f"https://opensupplyhub.org/facilities?q={q}&countries={code}&sort_by=contributors_desc"

# Apply to your dataframe
df_6m["osh_link"] = df_6m.apply(
    lambda r: make_osh_link(r["exporter_name"], r["exporter_country"]),
    axis=1
)


In [ ]:
df_6m["osh_link"].drop_duplicates().to_csv("output/osh_links.csv")

In [ ]:
df_6m["short_exporter_name"] = df_6m["exporter_name"].apply(clean_company_name)


In [ ]:
df_6m.short_exporter_name.drop_duplicates().to_csv("output/unique_exporter.csv")

In [ ]:
df_6m

In [ ]:
df_6m.exporter_name

In [ ]:
df_ted.winner_name.nunique()

In [ ]:

# Prepare for grouping
df = df_6m.copy()
df["year"] = df["publication_date"].dt.year
df["exporter_country"] = df["exporter_country"].fillna("Unknown")
df["winner_name"] = df["winner_name"].fillna("Unknown")

# Exporter ranking per winner-year
full_exporter_stats = (
    df.groupby(["winner_name", "year", "exporter_country"])
      .size()
      .reset_index(name="num_records")
      .sort_values(["winner_name", "year", "num_records"], ascending=[True, True, False])
)

full_exporter_stats["rank"] = (
    full_exporter_stats.groupby(["winner_name", "year"])["num_records"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# --- Aggregate to winner-level TOTAL country counts ---
winner_country_totals = (
    full_exporter_stats.groupby(["winner_name", "exporter_country"])["num_records"]
    .sum()
    .reset_index()
)

# --- Compute per-winner coverage threshold (15%) based on TOTAL ---
def compute_total_threshold(group, threshold=0):
    total = group["num_records"].sum()
    group = group.copy()
    group["pct"] = group["num_records"] / total

    selected = group[group["pct"] >= threshold].sort_values("pct", ascending=False)

    return pd.Series({
        "countries_over_15pct_list": selected["exporter_country"].tolist(),
        "countries_over_15pct_pct_list": selected["pct"].round(1).tolist(),  
        "countries_over_15pct_records_list": selected["num_records"].tolist(),
        "countries_over_15pct_count": len(selected),
        "total_countries": group["exporter_country"].nunique(),
    })


coverage_total = (
    winner_country_totals.groupby("winner_name")
    .apply(compute_total_threshold, threshold=0.1)
    .reset_index()
)

# Total lots per winner
total_lots_by_winner = (
    df_ted.groupby("winner_name")
          .size()
          .reset_index(name="total_winned_lots")
)

# CPV summary
lots_per_cpv = (
    df.groupby(["winner_name", "main_classification"])
      .size()
      .reset_index(name="lots_per_cpv")
)

lots_per_cpv_summary = (
    lots_per_cpv.groupby("winner_name")
    .apply(lambda g: dict(zip(g["main_classification"], g["lots_per_cpv"])))
    .reset_index(name="lots_per_cpv_dict")
)


lots_by_buyer_type = (
    df.groupby(["winner_name", "buyer_type"])
      .size()
      .reset_index(name="lots_per_group")
)

lots_by_buyer_summary = (
    lots_by_buyer_type.groupby("winner_name")
    .apply(lambda g: dict(zip(g["buyer_type"], g["lots_per_group"])))
    .reset_index(name="lots_per_buyer_group_dict")
)

# Records per winner-year
records_by_year = (
    full_exporter_stats.groupby(["winner_name", "year"])["num_records"]
    .sum()
    .reset_index()
)

records_by_year_list = (
    records_by_year.groupby("winner_name")
    .agg(
        years=("year", lambda x: sorted(x.unique())),
        records_by_year_list=("num_records", list)
    )
    .reset_index()
)


In [ ]:
# Extract TED year
df_ted["ted_year"] = df_ted["publication_date"].dt.year

# Lots per winner-year from TED
lots_by_year = (
    df_ted.groupby(["winner_name", "ted_year"])
          .size()
          .reset_index(name="lots_in_year")
)

# Convert to compact list form
lots_by_year_list = (
    lots_by_year.groupby("winner_name")
                .agg(
                    ted_years=("ted_year", lambda x: sorted(x.unique())),
                    lots_by_year_list=("lots_in_year", list)
                )
                .reset_index()
)

# Final combined table
final_df = (
    coverage_total  # already winner-level totals
    .rename(columns={
        "total_records": "trade_total_records",
        "total_countries": "trade_total_countries",
        "countries_over_15pct_list": "trade_countries_over_15pct_list",
        "countries_over_15pct_pct_list": "trade_countries_over_15pct_pct_list",
        "countries_over_15pct_records_list": "trade_countries_over_15pct_records_list",
        "countries_over_15pct_count": "trade_countries_over_15pct_count"
    })
    .merge(records_by_year_list.rename(columns={
        "years": "trade_years",
        "records_by_year_list": "trade_records_by_year_list"
    }), on="winner_name", how="left")
    .merge(total_lots_by_winner.rename(columns={
        "total_lots": "ted_total_lots"
    }), on="winner_name", how="left")
    .merge(lots_per_cpv_summary.rename(columns={
        "lots_per_cpv_dict": "ted_lots_per_cpv_dict"
    }), on="winner_name", how="left")
    .merge(lots_by_buyer_summary.rename(columns={
        "lots_per_buyer_group_dict": "ted_lots_per_buyer_group_dict"
    }), on="winner_name", how="left")
    .merge(lots_by_year_list, on="winner_name", how="left")
)

In [ ]:
final_df.total_winned_lots.sum()

In [ ]:
final_df

In [ ]:
cpv_map = {
    18000000: "General textiles / clothing accessories",
    18100000: "Occupational clothing",
    18200000: "Outerwear",
    18300000: "Garments",
    35200000: "Police equipment",
    39518000: "Hospital textile articles"
}


In [ ]:
def map_cpv(x):
    # already descriptive text → leave unchanged
    if isinstance(x, str) and not x.isdigit():
        return x

    # convert numeric strings → int
    try:
        key = int(x)
    except:
        return x

    # return mapped value if exists
    return cpv_map.get(key, x)


In [ ]:
df_ted["main_classification"] = df_ted["main_classification"].apply(map_cpv)


In [ ]:
# TED: extract classifications per winner
ted_classification = (
    df_ted.groupby("winner_name")["main_classification"]
          .apply(lambda x: sorted(set([str(i) for i in x if pd.notna(i)])))
          .reset_index(name="main_classification_list")
)


In [ ]:
# TRADE: extract HS codes per winner based on matched 6-month records
trade_hs = (
    df_6m.groupby("winner_name")["hs_code"]
         .apply(lambda x: sorted(set([str(i) for i in x if pd.notna(i)])))
         .reset_index(name="hs_code_list")
)


In [ ]:
final_df = (
    final_df
    .merge(ted_classification, on="winner_name", how="left")
    .merge(trade_hs, on="winner_name", how="left")
)


In [ ]:
final_df = final_df.rename(columns={"main_classification_list": "ted_main_classification_list"})
final_df = final_df.rename(columns={"hs_code_list": "trade_hs_code_list"})

# Convert to list safely (some entries may already be lists or dict keys)
def to_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return [x]

final_df["ted_main_classification_list"] = final_df["ted_main_classification_list"].apply(to_list)

# Remove winners containing unwanted textile types
unwanted = ["Chemical products", "Presents and rewards"]

final_filtered = final_df[
    ~final_df["ted_main_classification_list"].apply(
        lambda lst: any(item in lst for item in unwanted)
    )
].copy()

display(final_filtered)


In [ ]:
final_df.total_winned_lots.sum()

In [ ]:
df_plot = []

for _, row in final_filtered.iterrows():
    textile_types = row.get("ted_main_classification_list", [])
    countries = row.get("trade_countries_over_15pct_list", [])
    buyer_groups_dict = row.get("ted_lots_per_buyer_group_dict", {})
    lots = row.get("total_winned_lots", 0)

    # Skip empty rows
    if not isinstance(textile_types, list) or not isinstance(countries, list):
        continue
    if len(textile_types) == 0 or len(countries) == 0:
        continue

    # Expand buyer groups into list form
    if isinstance(buyer_groups_dict, dict):
        buyer_groups = list(buyer_groups_dict.keys())
    else:
        buyer_groups = []

    # If no buyer groups, mark as "unknown"
    if len(buyer_groups) == 0:
        buyer_groups = ["unknown"]

    # Expand combinations: textile × country × buyer_group
    for t in textile_types:
        for c in countries:
            for b in buyer_groups:
                df_plot.append({
                    "country": c,
                    "textile_type": t,
                    "buyer_type": b,
                    "lot_weight": lots
                })

df_plot = pd.DataFrame(df_plot)


In [ ]:
# Darker color palette
dark_palette = [
    "#C75D2C", "#972484", "#517fd6", "#0a5728",
    "#0f766e", "#1d4ed8", "#92822ada", "#92400e",
    "#b91c1c", "#065f46", "#78350f", "#312e81"
]


In [ ]:
plot_group_map = {

    "T-shirts": "T-shirts and shirts",
    "Socks": "Socks and Underwear",
    "Underwear": "Socks and Underwear",
    "Hospital textile articles": "Hospital linen"
}

In [ ]:
exclude_types = [
    "Hard hats",
    "Protective footwear",
    "Footwear",
    "Garments",
    "Trousers",
    "Miscellaneous services",
    "Services",
    "Visors",
    "Security",
]


In [ ]:
df_plot = df_plot[~df_plot["textile_type"].astype(str).isin(exclude_types)].copy()

In [ ]:
df_plot["textile_type"] = df_plot["textile_type"].replace(plot_group_map)


In [ ]:
df_plot["textile_type"].value_counts()

In [ ]:

# Aggregate after merge
df_grouped = (
    df_plot.groupby(["country", "textile_type"])["lot_weight"]
           .sum()
           .reset_index()
)

df_pivot = df_grouped.pivot(
    index="country",
    columns="textile_type",
    values="lot_weight"
).fillna(0)


fig = px.bar(
    df_pivot,
    x=df_pivot.index,
    y=df_pivot.columns,
    title="Countries Supplying Each Textile Type (Weighted by Total Lots Won)",
    labels={"value": "Total Weighted Lots", "country": "Country"},
)

fig.update_traces(
    textangle=0,              
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(size=16)
)
fig.update_layout(
    barmode="stack",
    xaxis_title="Exporter Country",
    yaxis_title="Total Weighted Lots",
    legend_title="Textile Type",
    width=1300,
    height=750,
    template="plotly_white",
    font=dict(size=13)
)

fig.show()


In [ ]:
colors = px.colors.qualitative.Safe_r

In [ ]:
df_grouped = (
    df_plot.groupby(["textile_type", "country"])["lot_weight"]
    .sum()
    .reset_index()
)

df_pivot = df_grouped.pivot(
    index="textile_type",
    columns="country",
    values="lot_weight"
).fillna(0)

# Light professional palette

fig = px.bar(
    df_pivot,
    x=df_pivot.index,
    y=df_pivot.columns,
    title="Exporting Countries per Textile Type (Weighted by Total Lots Won)",
    labels={"value": "Total Weighted Lots", "textile_type": "Textile Type"},
    color_discrete_sequence=colors,
    text_auto=True
)

fig.update_traces(
    textangle=0,              
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(size=16)
)

fig.update_layout(
    barmode="stack",
    xaxis_title="Textile Type",
    yaxis_title="Total Weighted Lots",
    legend_title="Country",
    width=1400,
    height=750,
    xaxis=dict(tickfont=dict(size=15), titlefont=dict(size=20)),
    yaxis=dict(tickfont=dict(size=15), titlefont=dict(size=20)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(color="#222", size=13),
    margin=dict(l=40, r=40, t=90, b=120)
)

# Rotate x-axis labels for readability
fig.update_xaxes(tickangle=45)

fig.show()

In [ ]:
df_buyer = df_plot[df_plot["buyer_type"].notna()].copy()

df_grouped_buyer = (
    df_buyer.groupby(["country", "buyer_type"])["lot_weight"]
            .sum()
            .reset_index()
)

df_pivot_buyer = df_grouped_buyer.pivot(
    index="country",
    columns="buyer_type",
    values="lot_weight"
).fillna(0)

fig_buyer = px.bar(
    df_pivot_buyer,
    x=df_pivot_buyer.index,
    y=df_pivot_buyer.columns,
    title="Public Institutions by Exporting Country (Weighted by Total Lots Won)",
    labels={"value": "Weighted Lots", "country": "Exporter Country"},
    color_discrete_sequence=dark_palette,
    text_auto=True
)

fig_buyer.update_traces(
    textangle=0,                 # horizontal
    textposition="inside",       # inside the bar
    insidetextanchor="middle"    ,    # centered inside segment
    textfont=dict(size=16)
)

fig_buyer.update_layout(
    barmode="stack",
    xaxis_title="Country",
    yaxis_title="Weighted Lots",
    legend_title="Buyer Type",
    width=1400,
    height=750,
    xaxis=dict(tickfont=dict(size=15), titlefont=dict(size=20)),
    yaxis=dict(tickfont=dict(size=15), titlefont=dict(size=20)),
    font=dict(color="#222", size=13)
)

fig_buyer.show()


In [ ]:
file_path = r"outputs/matched_ted_tradeatlas_6months_with_notices.xlsx"

df_6m.to_excel(file_path)

In [ ]:
def build_country_lot_table(final_df):
    """
    Convert final_df into a long table:
    - One row per (country, winner)
    - count = total_winned_lots
    """
    rows = []

    for _, row in final_df.iterrows():
        countries = row["trade_countries_over_15pct_list"]
        total_lots = row["total_winned_lots"]

        if not isinstance(countries, list):
            continue

        for country in countries:
            rows.append({
                "country": country,
                "count": total_lots
            })

    df_map = pd.DataFrame(rows)

    # Aggregate in case multiple winners link to same country
    df_map = df_map.groupby("country", as_index=False)["count"].sum()

    return df_map


df_lot_flows = build_country_lot_table(final_df)


In [ ]:
SPECIAL_ISO3 = {
    "Viet Nam": "VNM",
    "Vietnam": "VNM",
    "Russian Federation": "RUS",
    "Russia": "RUS",
    "Czechia": "CZE",
    "Ivory Coast": "CIV",
    "Côte d’Ivoire": "CIV",
    "Cote d'Ivoire": "CIV",
    "South Korea": "KOR",
    "Republic of Korea": "KOR",
    "North Macedonia": "MKD",
    "Türkiye": "TUR",
    "Turkey": "TUR",
}

def iso3_from_name(name: str):
    if pd.isna(name):
        return None
    name = str(name).strip()
    if name in SPECIAL_ISO3:
        return SPECIAL_ISO3[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None
    
def plot_lot_flow_map(df_map_input, html_path=None):
    df = df_map_input.copy()
    df["iso3"] = df["country"].apply(iso3_from_name)

    missing = df[df["iso3"].isna()]["country"].unique().tolist()
    if missing:
        print("⚠ Missing ISO3:", missing)

    df = df.dropna(subset=["iso3", "count"])
    df_targets = df[df["iso3"] != "DEU"]

    colorscale = ["#6a8bff", "#ff4ecb"]
    min_count = df_targets["count"].min()
    max_count = df_targets["count"].max()

    def line_color(value):
        t = (value - min_count) / (max_count - min_count + 1e-9)
        return colorscale[0] if t < 0.5 else colorscale[1]

    def line_width(value):
        return 1.0 + 3.0 * (value - min_count) / (max_count - min_count + 1e-9)

    fig = go.Figure()

    # flow lines
    for _, row in df_targets.iterrows():
        fig.add_trace(go.Scattergeo(
            locationmode="ISO-3",
            mode="lines",
            locations=["DEU", row["iso3"]],
            line=dict(
                width=line_width(row["count"]),
                color=line_color(row["count"])
            ),
            opacity=0.75,
            hoverinfo="text",
            text=f"Germany → {row['country']}<br>Lots: {row['count']}",
            showlegend=False
        ))

    # Germany node
    fig.add_trace(go.Scattergeo(
        locationmode="ISO-3",
        locations=["DEU"],
        mode="markers+text",
        marker=dict(size=18, color="#00d7ff", line=dict(color="white", width=1.2)),
        text=["GERMANY"],
        textposition="bottom center",
        textfont=dict(color="white", size=12),
        hoverinfo="skip",
        name="Germany"
    ))

    # destination nodes
    fig.add_trace(go.Scattergeo(
        locationmode="ISO-3",
        locations=df_targets["iso3"],
        mode="markers+text",
        marker=dict(
            size=(df_targets["count"] / df_targets["count"].max()) * 10 + 6,
            color="#ff4ecb",
            line=dict(color="white", width=0.7),
            opacity=0.95
        ),
        text=df_targets["iso3"],
        textposition="top center",
        textfont=dict(color="white", size=9),
        hoverinfo="skip"
    ))

    legend_text = "<br>".join(
        df_targets.sort_values("count", ascending=False)
        .apply(lambda r: f"{r['iso3']}: {r['count']}", axis=1)
        .tolist()
    )

    fig.update_layout(
        annotations=[
            dict(
                x=1.05, y=0.5, xref="paper", yref="paper",
                text=f"<b>Export Countries Weighted by Total Awarded Lots</b><br><br>{legend_text}",
                showarrow=False,
                font=dict(size=12, color="white"),
                align="left",
            )
        ],
        geo=dict(
            projection_type="natural earth",
            showland=True,
            landcolor="#05070c",
            countrycolor="#444",
            coastlinecolor="#444",
            showcountries=True,
            oceancolor="#02040a",
            showocean=True,
        ),
        paper_bgcolor="#000000",
        plot_bgcolor="#000000",
        height=850,
        width=1900,
        margin=dict(l=0, r=250, t=80, b=0),
        showlegend=False
    )

    if html_path:
        os.makedirs(os.path.dirname(html_path), exist_ok=True)
        fig.write_html(html_path, include_plotlyjs="cdn")
        print("Saved ➜", html_path)

    return fig


In [ ]:
df_map = build_country_lot_table(final_df)

plot_lot_flow_map(df_map, html_path="output/lots_export_map.html")
